# Figure G04: missed-gene candidates against the matched clean-shadow comparator (GTDB r232)

**Status:** draft, Gadi. The ID is provisional and the integrator renumbers it in the figure plan.

## Provenance

| plotting file | built from | producing stage |
|---|---|---|
| `data/figG04_funnel.tsv.gz` | per-chunk `missed_genes/{bac,arc}/*.stats.tsv` (760 + 41 chunks) on Gadi `/g/data/ob80/re3494/gtdb_entropy/` | `Validation/gtdb/scripts/10_missed_genes.py` |
| `data/figG04_matched_comparator.tsv.gz` | `Validation/gtdb/matched_comparator/{bac,arc}_v7_matched.tsv` (sha256 `376ec31e…`, `ed5b5c22…`) | `Validation/gtdb/scripts/41_comparator_summary.py`, v7 matched comparator (genome_entropy #109, #111, #113, #114) |

- **Source:** `linsalrob/genome_entropy` `main` at `97a792f`.
- **Reduction:** `manuscript/prep/gtdb_prepare_data.py --only G04`. It sums the chunk counts and copies the 41 summary verbatim. Nothing is re-estimated.
- **Definitions** (`Validation/gtdb/scientific_report.md` §6, `matched_comparator.md`):
  - A **candidate** is an ORF in an annotated genome that matches no GenBank CDS, has 3Di entropy ≥ 2.5 bits, and is not a shadow of an annotated CDS.
  - The **comparator** is one antisense or same-strand-frameshift shadow per candidate. Comparators are matched without replacement on truncation status, 3Di (±0.025 bits) and length (±10%), with seed 0. 817 bacterial and 972 archaeal draws were made with replacement, and every comparator rate is weighted by draw multiplicity.
  - A hit counts as **full-length** when Foldseek structural search (`-e 0.001`) gives qcov ≥ 0.8 and tcov ≥ 0.8.
  - **Implied real-gene share** = (candidate − comparator) / (annotated − comparator), with a 95% CI. BFVD is shown but not used in the estimate, because its annotated ceiling is only about 11%.
  - The **ladder excess** is candidates minus the scaled comparator. It is a floor on the number of real genes, not an estimate of it.


In [ ]:
import gzip
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker

DATA = Path("../data")
IMAGES = Path("../images")
IMAGES.mkdir(exist_ok=True)

# Okabe-Ito, colour-blind safe. One colour per arm throughout the GTDB figures.
ARM = {"candidate": "#0072B2", "comparator": "#D55E00",
       "annotated": "#009E73", "intergenic": "#999999"}
DOMAIN = {"bacteria": "#0072B2", "archaea": "#56B4E9"}
INK = "#222222"
MUTED = "#555555"

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5,
    "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": INK, "axes.labelcolor": INK,
    "xtick.color": INK, "ytick.color": INK,
    "svg.hashsalt": "genome_entropy", "pdf.fonttype": 42,
})


def panel(ax, letter):
    ax.text(-0.02, 1.04, letter, transform=ax.transAxes, fontsize=10,
            fontweight="bold", ha="right", va="bottom")


def save(fig, stem):
    # No timestamps or software strings: reruns are byte-stable.
    fig.savefig(IMAGES / f"{stem}.pdf", bbox_inches="tight",
                metadata={"CreationDate": None, "ModDate": None,
                          "Producer": None, "Creator": None})
    fig.savefig(IMAGES / f"{stem}.png", dpi=300, bbox_inches="tight",
                metadata={"Software": None})
    print("wrote", IMAGES / f"{stem}.pdf", IMAGES / f"{stem}.png")


def matched(path):
    """Long table from 41_comparator_summary.py -> {(domain, metric): (v, lo, hi)}."""
    d = pd.read_csv(path, sep="\t")
    return {(r.domain_name, r.metric): (r.value, r.lo, r.hi) for r in d.itertuples()}


In [ ]:
m = matched(DATA / "figG04_matched_comparator.tsv.gz")
funnel = pd.read_csv(DATA / "figG04_funnel.tsv.gz", sep="\t")
DOMS = ["bacteria", "archaea"]
DBS = ["afdb_swissprot", "pdb100", "cath50", "bfvd"]
DB_LABEL = {"afdb_swissprot": "Swiss-Prot", "pdb100": "PDB100",
            "cath50": "CATH50", "bfvd": "BFVD"}
LADDER = {"C3": "any full-length hit", "C4": "full-length in ≥2 DBs",
          "C5": "full-length Swiss-Prot/PDB/CATH", "C6": "structure-only, full-length",
          "C7": "strict combined rule"}

fig = plt.figure(figsize=(7.2, 9.0))
gs = fig.add_gridspec(4, 2, height_ratios=[1.0, 1.15, 0.9, 1.05],
                      hspace=0.85, wspace=0.55)

# a: funnel
for j, dom in enumerate(DOMS):
    ax = fig.add_subplot(gs[0, j])
    f = funnel[funnel.domain == dom].sort_values("step_order")
    y = np.arange(len(f))[::-1]
    cols = ["#c9c5bf"] * (len(f) - 1) + [ARM["candidate"]]
    ax.barh(y, f.n_orfs, color=cols, height=0.62)
    ax.set_xscale("log")
    ax.set_yticks(y, f.step if j == 0 else [""] * len(f))
    ax.set_xlabel("ORFs (log scale)")
    ax.set_title(dom, loc="left")
    for yy, v in zip(y, f.n_orfs):
        ax.text(v * 1.4, yy, f"{v:,}", va="center", fontsize=6.4, color=INK)
    ax.set_xlim(1e3 if dom == "archaea" else 1e4, f.n_orfs.max() * 40)
    if j == 0:
        panel(ax, "a")

# b: full-length coverage per arm and database
for j, dom in enumerate(DOMS):
    ax = fig.add_subplot(gs[1, j])
    x = np.arange(len(DBS))
    w = 0.26
    for k, (arm, key, lbl) in enumerate([
            ("candidate", "candidate", "candidates"),
            ("comparator", "comparator", "matched clean shadows"),
            ("annotated", "annotated", "annotated CDS")]):
        v = [m[(dom, f"fulllen_{db}_{key}")][0] * 100 for db in DBS]
        ax.bar(x + (k - 1) * w, v, width=w, color=ARM[arm], label=lbl)
    ax.set_xticks(x, [DB_LABEL[d] for d in DBS])
    ax.set_ylabel("% with full-length hit")
    ax.set_title(f"{dom}  (n = {int(m[(dom, 'n_candidate')][0]):,} per arm)", loc="left")
    ax.set_ylim(0, 40)
    if j == 0:
        ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.10), ncol=3, frameon=False)
        panel(ax, "b")

# c: implied real-gene share, both domains in one panel
ax = fig.add_subplot(gs[2, :])
est = ["afdb_swissprot", "pdb100", "cath50", "any_sp_pdb_cath"]
lab = ["Swiss-Prot", "PDB100", "CATH50", "any of the three"]
w = 0.36
for k, dom in enumerate(DOMS):
    v = np.array([m[(dom, f"share_{e}")] for e in est], dtype=float) * 100
    xs = np.arange(len(est)) + (k - 0.5) * w
    ax.bar(xs, v[:, 0], width=w, color=DOMAIN[dom], label=dom,
           yerr=[v[:, 0] - v[:, 1], v[:, 2] - v[:, 0]],
           error_kw=dict(ecolor=MUTED, lw=0.8, capsize=2))
    for xx, (val, lo, hi) in zip(xs, v):
        ax.text(xx, hi + 0.8, f"{val:.1f}", ha="center", fontsize=6.4, color=INK)
ax.set_xticks(np.arange(len(est)), lab)
ax.set_ylabel("implied real-gene share (%)\n95% CI")
ax.set_ylim(0, 32)
ax.legend(frameon=False, loc="upper left")
se = {d: m[(d, "structural_estimate")] for d in DOMS}
panel(ax, "c")

# d: evidence ladder, candidates vs scaled comparator
for j, dom in enumerate(DOMS):
    ax = fig.add_subplot(gs[3, j])
    cand = np.array([m[(dom, f"ladder_{c}_candidates")][0] for c in LADDER])
    scal = np.array([m[(dom, f"ladder_{c}_comparator_scaled")][0] for c in LADDER])
    exc = np.array([m[(dom, f"ladder_{c}_excess")][0] for c in LADDER])
    y = np.arange(len(LADDER))[::-1]
    h = 0.36
    ax.barh(y + h / 2, cand, height=h, color=ARM["candidate"], label="candidates")
    ax.barh(y - h / 2, scal, height=h, color=ARM["comparator"],
            label="matched clean shadows (weighted)")
    ax.set_yticks(y, list(LADDER.values()) if j == 0 else [""] * len(LADDER))
    ax.set_xlabel("ORFs")
    ax.set_title(dom, loc="left")
    for yy, c, e in zip(y, cand, exc):
        ax.text(cand.max() * 1.03, yy, f"+{e:,.0f}", va="center", fontsize=6.4, color=INK)
    ax.set_xlim(0, cand.max() * 1.38)
    ax.xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(
        lambda v, _: f"{v/1000:g}k" if v else "0"))
    ax.xaxis.set_major_locator(matplotlib.ticker.MaxNLocator(4))
    if j == 0:
        panel(ax, "d")
        ax.legend(loc="lower center", bbox_to_anchor=(1.15, -0.62), ncol=2, frameon=False)

save(fig, "figG04_missed_genes")


In [ ]:
# Numbers the manuscript text quotes from this figure; printed so text and figure can be cross-checked.
for dom in DOMS:
    print(dom,
          "candidates", f"{int(m[(dom, 'n_candidate')][0]):,}",
          "| Swiss-Prot share", "{:.1%} ({:.1%}-{:.1%})".format(*m[(dom, 'share_afdb_swissprot')]),
          "| structural estimate", "{:,.0f}-{:,.0f}".format(*m[(dom, 'structural_estimate')][1:]) if not np.isnan(m[(dom, 'structural_estimate')][1]) else m[(dom, 'structural_estimate')],
          "| C3 excess", f"{m[(dom, 'ladder_C3_excess')][0]:,.0f}",
          "| C7 excess", f"{m[(dom, 'ladder_C7_excess')][0]:,.0f}")


## Caption (draft)

**Figure G04. Six-frame ORF calls that GenBank misses carry real genes in excess over a matched non-coding comparator (GTDB r232).**

**(a)** The candidate funnel, from every six-frame ORF call (bacteria 2,568,244,984 from 189,715 genomes; archaea 54,858,398 from 10,122 genomes) to ORFs in annotated genomes that are unmatched by GenBank, have 3Di entropy ≥ 2.5 bits and are not shadows of an annotated CDS: 523,346 bacterial and 22,447 archaeal candidates.

**(b)** The percentage of each arm with a full-length (qcov and tcov ≥ 0.8) Foldseek structural hit in each database. The arms are the candidates, an equal number of antisense or frameshifted shadows matched to the candidates on length, 3Di entropy and contig truncation, and annotated CDS as the positive control.

**(c)** The implied real-gene share, (candidate − comparator) / (annotated − comparator), with 95% confidence intervals. BFVD is excluded because annotated CDS reach only about 11% there.

**(d)** Candidates meeting each evidence criterion, against the weighted, scaled matched comparator. The labels give the excess, which is a conservative floor on the number of real genes.

In both domains candidates lie between their matched shadows and annotated genes. Comparator rates are weighted by draw multiplicity (0.16% of bacterial and 4.3% of archaeal draws were with replacement).
